# GCN Training — Configuration, Evaluation, Checkpoints

This notebook focuses on the **engineering workflow**: loading config, training with reproducibility, saving/loading checkpoints, and extracting embeddings for downstream use.

The implementation is imported from `src/vanilla_gcn/` — no model code lives in this notebook.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

from vanilla_gcn.config import load_config
from vanilla_gcn.seed import set_seed
from vanilla_gcn.data.synthetic import create_synthetic_graph
from vanilla_gcn.data.preprocessing import prepare_graph
from vanilla_gcn.models.vanilla_gcn import VanillaGCN
from vanilla_gcn.training.trainer import train_gcn
from vanilla_gcn.training.evaluation import evaluate_gcn
from vanilla_gcn.utils.checkpointing import save_checkpoint, load_checkpoint
from vanilla_gcn.utils.logging import setup_logging
from vanilla_gcn.visualization.embeddings import visualize_embeddings, visualize_training_curves
from vanilla_gcn.visualization.graph import visualize_predictions

setup_logging('INFO', force=True)
print('Imports OK')

## 1. Load Configuration

All hyperparameters come from `configs/default.yaml`. Nothing is hard-coded.

In [ ]:
cfg = load_config(ROOT / 'configs' / 'default.yaml')
print('Seed         :', cfg.seed)
print('Hidden dim   :', cfg.model.hidden_dim)
print('Num layers   :', cfg.model.num_layers)
print('Dropout      :', cfg.model.dropout)
print('LR           :', cfg.training.learning_rate)
print('Weight decay :', cfg.training.weight_decay)
print('Epochs       :', cfg.training.epochs)
print('Train ratio  :', cfg.data.train_ratio)

## 2. Data + Preprocessing

In [ ]:
set_seed(cfg.seed)
data = create_synthetic_graph(
    seed=cfg.seed,
    train_ratio=cfg.data.train_ratio,
    val_ratio=cfg.data.validation_ratio,
    test_ratio=cfg.data.test_ratio,
)
print(data)

A_hat, A_tilde, X, y = prepare_graph(data.adjacency, data.features, data.labels)
print('\nPreprocessing complete.')
print(f'Ã shape: {tuple(A_tilde.shape)}')

## 3. Build and Train Model

In [ ]:
set_seed(cfg.seed)
model = VanillaGCN(
    input_dim=data.num_features,
    hidden_dim=cfg.model.hidden_dim,
    num_classes=data.num_classes,
    num_layers=cfg.model.num_layers,
    dropout=cfg.model.dropout,
    debug=False,
)
model.print_summary()

history = train_gcn(
    model, A_tilde, X, y,
    data.train_mask, data.val_mask,
    epochs=cfg.training.epochs,
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
)
print(history.summary())

## 4. Training Curves

In [ ]:
fig = visualize_training_curves(
    history.train_loss, history.val_loss,
    history.train_acc, history.val_acc,
    history.epochs,
    save_path=ROOT / 'outputs/figures/02_training_curves.png',
)
plt.show()

## 5. Evaluation

In [ ]:
metrics = evaluate_gcn(
    model, A_tilde, X, y,
    data.train_mask, data.val_mask, data.test_mask,
)
print(metrics.summary())

## 6. Save Checkpoint

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.training.learning_rate,
                       weight_decay=cfg.training.weight_decay)
ckpt_path = ROOT / 'outputs' / 'checkpoints' / 'notebook02_model.pt'
save_checkpoint(model, optimizer, cfg, history,
                epoch=cfg.training.epochs, path=ckpt_path)
print(f'Saved to {ckpt_path}')

## 7. Load Checkpoint and Verify Identical Output

In [ ]:
# Original logits
model.eval()
with torch.no_grad():
    logits_original = model(X, A_tilde)

# Fresh model + load
model2 = VanillaGCN(
    input_dim=data.num_features,
    hidden_dim=cfg.model.hidden_dim,
    num_classes=data.num_classes,
    num_layers=cfg.model.num_layers,
)
optimizer2 = optim.Adam(model2.parameters())
model2, optimizer2, saved_cfg, loaded_hist, epoch = load_checkpoint(ckpt_path, model2, optimizer2)

model2.eval()
with torch.no_grad():
    logits_loaded = model2(X, A_tilde)

match = torch.allclose(logits_original, logits_loaded, atol=1e-6)
print(f'Logits identical after load: {match}')
print(f'Loaded from epoch: {epoch}')

## 8. Embedding Extraction

Extract the final hidden representations $Z$ (before the output layer).

In [ ]:
embeddings = model.get_embeddings(X, A_tilde)
print('Available embedding layers:', list(embeddings.keys()))

# Get the last hidden layer (penultimate)
hidden_keys = [k for k in embeddings if k not in ('input', 'logits')]
Z = embeddings[hidden_keys[-1]] if hidden_keys else embeddings['input']
print(f'Final hidden embeddings Z shape: {tuple(Z.shape)}')

fig = visualize_embeddings(
    Z, data.labels, data.node_ids,
    title='Final Node Embeddings Z (PCA)',
    save_path=ROOT / 'outputs/figures/02_embeddings.png',
)
plt.show()

## 9. Reproducibility Test

Two runs with the same seed must produce identical results.

In [ ]:
def run_with_seed(seed_val, epochs=20):
    set_seed(seed_val)
    m = VanillaGCN(data.num_features, cfg.model.hidden_dim,
                   data.num_classes, cfg.model.num_layers, debug=False)
    train_gcn(m, A_tilde, X, y, data.train_mask, data.val_mask,
              epochs=epochs, lr=0.01, log_every=epochs)
    m.eval()
    with torch.no_grad():
        return m(X, A_tilde)

out_a = run_with_seed(42)
out_b = run_with_seed(42)
print(f'Run A == Run B (seed=42): {torch.allclose(out_a, out_b, atol=1e-6)}')

out_c = run_with_seed(123)
print(f'Run A != Run C (seed=123): {not torch.allclose(out_a, out_c, atol=1e-6)}')

## 10. Final Predictions Graph

In [ ]:
fig = visualize_predictions(
    data.adjacency, data.labels, metrics.predictions, data.node_ids,
    save_path=ROOT / 'outputs/figures/02_predictions.png',
)
plt.show()